# Implied Vol Fit

Author: Sebastien Gurrieri, sebgur@gmail.com

 [[Gurrieri 2010]](    https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1779463)


### Retrieving Market Data

In [ ]:
# Import relevant modules
from datetime import datetime as dt
import numpy as np
from platform import python_version
import matplotlib.pyplot as plt
import pandas as pd

import sdevpy as sd

print("Python version: " + python_version())
print("NumPy version: " + np.__version__)
print("SDevPy version: " + sd.__version__)

Python version: 3.10.12
NumPy version: 1.26.4
TensorFlow version: 2.17.0
SDevPy version: 1.0.3


In [ ]:
name = "ABC"
valdate = dt.datetime(2025, 12, 15)

# Retrieve target market option data
file = vsurf.data_file(vsurf.test_data_folder(), name, valdate)
surface_data = vsurf.eqvolsurfacedata_from_file(file)
expiries = surface_data.expiries
fwds = surface_data.forwards
strike_surface = surface_data.get_strikes('absolute')
vol_surface = surface_data.vols

# Set calibration time grid
expiry_grid = np.array([timegrids.model_time(valdate, expiry) for expiry in expiries])

# Set calibration targets
# cf_price_surface, ftols = calibration_targets(expiry_grid, fwds, strike_surface, vol_surface)

# Reformat inputs to flat vectors
t, k, f, s = [], [], [], []
for i in range(len(expiries)):
    expiry = expiry_grid[i]
    fwd = fwds[i]
    strikes = strike_surface[i]
    vols = vol_surface[i]
    for strike, vol in zip(strikes, vols, strict=True):
        t.append(expiry)
        f.append(fwd)
        k.append(strike)
        s.append(vol)

t = np.asarray(t)
k = np.asarray(k)
f = np.asarray(f)
s = np.asarray(s)
is_call = True

# Initialize model
surface = TsSvi1()
# params_init = [0.20, 0.25, 0.10, 2.5, 0.1, 0.1, 0.9, 0.1, 0.1, 0.1, 0.1]
params_init = surface.initial_point()
surface.update_params(params_init)

# Optimizer settings
method = 'SLSQP' # L-BFGS-B, SLSQP, DE
tol = 1e-6

# Constraints
bounds = surface.bounds()

# Objective
obj_builder = TsSvi1ObjectiveBuilder(surface, t, k, f, s, config={})
objective = obj_builder.objective

# Optimize
optimizer = create_optimizer(method, tol=tol)
result = optimizer.minimize(objective, x0=params_init, bounds=bounds)
sol = result.x # Optimum parameters

# Calculate model vols and reshape results per maturity
# sol = params_init
surface.update_params(sol)
surface.check_params()
z = surface.calculate(t, k, is_call, f)
print(f"Result shape: {z.shape}")
model_vols = []
counter = 0
for _ in expiries:
    vols = []
    for _ in strikes:
        vols.append(z[counter])
        counter += 1
    model_vols.append(vols)



In [ ]:
# Estimate model on points and calculate RMSE, plot comparison
n_rows, n_cols = 3, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 8))
for i in range(n_rows):
    for j in range(n_cols):
        ax = axes[i, j]
        exp_idx = n_cols * i + j
        expiry = expiry_grid[exp_idx]
        fwd = fwds[exp_idx]
        strikes = strike_surface[exp_idx]
        min_k, max_k = strikes[0], strikes[-1]
        m_strikes = np.linspace(0.8 * min_k, 1.2 * max_k, 100)
        m_vols = surface.calculate(expiry, m_strikes, is_call, fwd)
        ax.scatter(strikes, vol_surface[exp_idx], label="market", color='black')
        ax.plot(m_strikes, m_vols, label="model", color='green')
        vol_rmse = rmse(vol_surface[exp_idx], model_vols[exp_idx])
        ax.set_title(f"T:{expiry:.2f}, RMSE(bps): {10000.0 * vol_rmse:,.2f}")
        ax.set_xlabel('strike')
        ax.set_ylabel('vol')
        ax.legend()

fig.suptitle('Option vols, Model vs Market', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()
